# Statistical Machine Learning (SML) Homework 4: Kernel K-Means Clustering

**Topic:** Kernel K-Means & Non-linear Clustering  
**Goal:** Implement, analyze, and visualize Kernel K-Means on complex non-linearly separable datasets, demonstrating mathematical understanding and optimization tricks.  

---

### Executive Summary
<div dir="rtl">

در الگوریتم K-Means استاندارد، فرض بر این است که خوشه‌ها به صورت خطی جداپذیر، کروی و هم‌اندازه هستند، زیرا از فاصله اقلیدسی استفاده می‌شود. هنگام مواجهه با ساختارهای پیچیده و غیرخطی (مانند دایره‌های هم‌مرکز یا هلال‌های درهم‌تنیده)، K-Means استاندارد کاملاً شکست می‌خورد.

</div>

<div dir="rtl">

**Kernel K-Means** این محدودیت را با نگاشت ضمنی داده‌ها به یک فضای ویژگی با بُعد بالا $\mathcal{H}$ با استفاده از یک تابع نگاشت غیرخطی $\Phi(x)$ برطرف می‌کند. با به‌کارگیری **ترفند کرنل**، می‌توانیم تخصیص خوشه‌ها را در $\mathcal{H}$ تنها با استفاده از ضرب‌های داخلی (از طریق تابع کرنل $K(x, y) = \langle\Phi(x), \Phi(y)\rangle$) محاسبه کنیم، بدون اینکه هرگز مختصات بُعد بالا را به صورت صریح محاسبه کنیم.

</div>

<div dir="rtl">

این نوت‌بوک شامل موارد زیر است:

</div>

1. **Rigorous Mathematical Derivations** <span dir="rtl">از تابع هدف Kernel K-means و محاسبه فاصله.</span>
2. **An Object-Oriented Implementation** <span dir="rtl">از Kernel K-means (سازگار با scikit-learn).</span>
3. **Comparative Analysis** <span dir="rtl">مقایسه K-Means استاندارد با Kernel K-Means روی مجموعه‌داده‌های مصنوعی متعدد.</span>
4. **Hyperparameter Sensitivity Studies** <span dir="rtl">نشان‌دهنده تأثیر پارامتر $\gamma$ کرنل RBF.</span>

## 1. Mathematical Foundations of Kernel K-Means

<div dir="rtl">

فرض کنید $X = \{x_1, x_2, \dots, x_N\}$ یک مجموعه‌داده شامل $N$ نمونه در $\mathbb{R}^d$ باشد. هدف ما افراز داده‌ها به $K$ خوشه مجزا $\{C_1, C_2, \dots, C_K\}$ است.

</div>

### 1.1 Mapping to Feature Space
<div dir="rtl">

فرض کنید $\Phi: \mathbb{R}^d \to \mathcal{H}$ یک تابع نگاشت غیرخطی باشد که نمونه‌ها را به یک فضای هیلبرت $\mathcal{H}$ با بُعد بالا (و احتمالاً بی‌نهایت‌بُعدی) تصویر می‌کند.

در این فضای بُعد بالا، هدف Kernel K-Means کمینه‌سازی مجموع مربعات فاصله‌ها تا مراکز خوشه‌ها است:

</div>

$$
J = \sum_{c=1}^K \sum_{x_i \in C_c} \| \Phi(x_i) - \mu_c \|^2
$$

<div dir="rtl">

که در آن $\mu_c$ مرکز خوشه $C_c$ در فضای ویژگی است:

</div>

$$
\mu_c = \frac{1}{|C_c|} \sum_{x_j \in C_c} \Phi(x_j)
$$

### 1.2 The Kernel Trick Derivation
<div dir="rtl">

از آنجا که $\mathcal{H}$ ممکن است بسیار بُعد بالا یا بی‌نهایت‌بُعدی باشد، نمی‌توانیم $\Phi(x_i)$ یا $\mu_c$ را مستقیماً محاسبه کنیم. در عوض، از یک تابع کرنل $K(x_i, x_j) = \langle \Phi(x_i), \Phi(x_j) \rangle$ برای محاسبه فاصله‌ها استفاده می‌کنیم.

بیایید مربع فاصله بین نقطه نگاشت‌شده $\Phi(x_i)$ و مرکز خوشه $\mu_c$ را بسط دهیم:

</div>

$$
\| \Phi(x_i) - \mu_c \|^2 = \langle \Phi(x_i) - \mu_c, \Phi(x_i) - \mu_c \rangle
$$
$$
\| \Phi(x_i) - \mu_c \|^2 = \langle \Phi(x_i), \Phi(x_i) \rangle - 2 \langle \Phi(x_i), \mu_c \rangle + \langle \mu_c, \mu_c \rangle
$$

<div dir="rtl">

حال $\mu_c = \frac{1}{|C_c|} \sum_{x_j \in C_c} \Phi(x_j)$ را در جملات جایگذاری می‌کنیم:

</div>

1. **First Term**:
   $$
   \langle \Phi(x_i), \Phi(x_i) \rangle = K(x_i, x_i)
   $$

2. **Second Term**:
   $$
   -2 \langle \Phi(x_i), \mu_c \rangle = -2 \left\langle \Phi(x_i), \frac{1}{|C_c|} \sum_{x_j \in C_c} \Phi(x_j) \right\rangle = -\frac{2}{|C_c|} \sum_{x_j \in C_c} K(x_i, x_j)
   $$

3. **Third Term**:
   $$
   \langle \mu_c, \mu_c \rangle = \left\langle \frac{1}{|C_c|} \sum_{x_j \in C_c} \Phi(x_j), \frac{1}{|C_c|} \sum_{x_l \in C_c} \Phi(x_l) \right\rangle = \frac{1}{|C_c|^2} \sum_{x_j \in C_c} \sum_{x_l \in C_c} K(x_j, x_l)
   $$

<div dir="rtl">

با کنار هم قرار دادن همه اینها، مربع فاصله در فضای ویژگی برابر است با:

</div>

$$
d^2(\Phi(x_i), C_c) = K(x_i, x_i) - \frac{2}{|C_c|} \sum_{x_j \in C_c} K(x_i, x_j) + \frac{1}{|C_c|^2} \sum_{x_j \in C_c} \sum_{x_l \in C_c} K(x_j, x_l)
$$

### 1.3 Key Optimization Trick (Omitting $K(x_i, x_i)$)
<div dir="rtl">

در مرحله تخصیص خوشه، هر نمونه $x_i$ به خوشه‌ای اختصاص داده می‌شود که فاصله را کمینه کند:

</div>

$$
c^*(x_i) = \arg\min_{c} d^2(\Phi(x_i), C_c)
$$

<div dir="rtl">

توجه کنید که $K(x_i, x_i)$ نسبت به اندیس خوشه $c$ ثابت است! این مقدار فقط به نمونه $x_i$ بستگی دارد. بنابراین:

</div>

$$
\arg\min_{c} d^2(\Phi(x_i), C_c) = \arg\min_{c} \left( -\frac{2}{|C_c|} \sum_{x_j \in C_c} K(x_i, x_j) + \frac{1}{|C_c|^2} \sum_{x_j \in C_c} \sum_{x_l \in C_c} K(x_j, x_l) \right)
$$

<div dir="rtl">

این یک بهینه‌سازی کلیدی است. این امکان را فراهم می‌کند که $K(x_i, x_i)$ را به طور کامل از محاسبات ماتریس فاصله در حین خوشه‌بندی حذف کنیم، که سرعت و کارایی عددی را به شدت بهبود می‌بخشد.

</div>

## 2. Python Implementation of Kernel K-Means

<div dir="rtl">

در زیر پیاده‌سازی کامل و شیءگرای الگوریتم خوشه‌بندی `KernelKMeans` که با scikit-learn سازگار است، ارائه شده است.

</div>

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.utils import check_random_state

class KernelKMeans(BaseEstimator, ClusterMixin):
    """
    Kernel K-means

    Reference
    ---------
    Kernel k-means, Spectral Clustering and Normalized Cuts.
    Inderjit S. Dhillon, Yuqiang Guan, Brian Kulis.
    KDD 2004.
    """

    def __init__(self, n_clusters=3, max_iter=50, tol=1e-3, random_state=None,
                 kernel="linear", gamma=None, degree=3, coef0=1,
                 kernel_params=None, verbose=0):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.kernel_params = kernel_params
        self.verbose = verbose

    @property
    def _pairwise(self):
        return self.kernel == "precomputed"

    def _get_kernel(self, X, Y=None):
        if callable(self.kernel):
            params = self.kernel_params or {}
        else:
            params = {"gamma": self.gamma,
                      "degree": self.degree,
                      "coef0": self.coef0}
        return pairwise_kernels(X, Y, metric=self.kernel,
                                filter_params=True, **params)

    def fit(self, X, y=None, sample_weight=None):
        n_samples = X.shape[0]

        K = self._get_kernel(X)

        sw = sample_weight if sample_weight is not None else np.ones(n_samples)
        self.sample_weight_ = sw

        rs = check_random_state(self.random_state)
        self.labels_ = rs.randint(self.n_clusters, size=n_samples)

        dist = np.zeros((n_samples, self.n_clusters))
        self.within_distances_ = np.zeros(self.n_clusters)

        for it in range(self.max_iter):
            dist.fill(0)
            self._compute_dist(K, dist, self.within_distances_,
                               update_within=True)
            labels_old = self.labels_
            self.labels_ = dist.argmin(axis=1)

            # Compute the number of samples whose cluster did not change
            # since last iteration.
            n_same = np.sum((self.labels_ - labels_old) == 0)
            if 1 - float(n_same) / n_samples < self.tol:
                if self.verbose:
                    print()
                    print("Converged at iteration", it + 1)
                break

        self.X_fit_ = X

        return self

    def _compute_dist(self, K, dist, within_distances, update_within):
        """Compute a n_samples x n_clusters distance matrix using the
        kernel trick."""
        sw = self.sample_weight_

        for j in range(self.n_clusters):
            mask = self.labels_ == j

            if np.sum(mask) == 0:
                raise ValueError("Empty cluster found, try smaller n_clusters or different initialization.")

            denom = sw[mask].sum()
            denomsq = denom * denom

            if update_within:
                KK = K[mask][:, mask]
                dist_j = np.sum(np.outer(sw[mask], sw[mask]) * KK / denomsq)
                within_distances[j] = dist_j
                dist[:, j] += dist_j
            else:
                dist[:, j] += within_distances[j]

            dist[:, j] -= 2 * np.sum(sw[mask] * K[:, mask], axis=1) / denom

    def predict(self, X):
        K = self._get_kernel(X, self.X_fit_)
        n_samples = X.shape[0]
        dist = np.zeros((n_samples, self.n_clusters))
        self._compute_dist(K, dist, self.within_distances_,
                           update_within=False)
        return dist.argmin(axis=1)

## 3. Comparative Experiments and Visualizations

<div dir="rtl">

بیایید توابع کمکی رسم نمودار و ارزیابی را برای مقایسه `KernelKMeans` با `KMeans` استاندارد scikit-learn بنویسیم.

</div>

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score

def plot_clustering_results(X, y_true, y_std, y_kernel, title, kernel_name="Kernel K-Means"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    
    # Ground Truth
    axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', edgecolor='k', alpha=0.8, s=50)
    axes[0].set_title("Ground Truth Labels", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Feature 1", fontsize=10)
    axes[0].set_ylabel("Feature 2", fontsize=10)
    axes[0].grid(True, linestyle='--', alpha=0.5)
    
    # Standard K-Means
    ari_std = adjusted_rand_score(y_true, y_std)
    sil_std = silhouette_score(X, y_std) if len(np.unique(y_std)) > 1 else 0.0
    axes[1].scatter(X[:, 0], X[:, 1], c=y_std, cmap='plasma', edgecolor='k', alpha=0.8, s=50)
    axes[1].set_title(f"Standard K-Means\nARI: {ari_std:.4f} | Silh: {sil_std:.4f}", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Feature 1", fontsize=10)
    axes[1].set_ylabel("Feature 2", fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    # Kernel K-Means
    ari_kernel = adjusted_rand_score(y_true, y_kernel)
    sil_kernel = silhouette_score(X, y_kernel) if len(np.unique(y_kernel)) > 1 else 0.0
    axes[2].scatter(X[:, 0], X[:, 1], c=y_kernel, cmap='coolwarm', edgecolor='k', alpha=0.8, s=50)
    axes[2].set_title(f"{kernel_name}\nARI: {ari_kernel:.4f} | Silh: {sil_kernel:.4f}", fontsize=13, fontweight='bold')
    axes[2].set_xlabel("Feature 1", fontsize=10)
    axes[2].set_ylabel("Feature 2", fontsize=10)
    axes[2].grid(True, linestyle='--', alpha=0.5)
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

### 3.1 Experiment 1: Linearly Separable Blobs

<div dir="rtl">

سه توده گاوسی مجزا و همسانگرد تولید می‌کنیم. از آنجا که این خوشه‌ها به صورت خطی جداپذیر هستند، K-Means استاندارد به راحتی آنها را مدیریت می‌کند. Kernel K-Means با **کرنل خطی** ($K(x, y) = x^T y$) باید عملکرد یکسانی داشته باشد زیرا نگاشت ضمنی، نگاشت همانی $\Phi(x) = x$ است.

</div>

In [ ]:
# Generate Blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Fit Standard K-Means
km_std = KMeans(n_clusters=3, random_state=42, n_init='auto')
y_std_blobs = km_std.fit_predict(X_blobs)

# Fit Kernel K-Means with Linear Kernel
km_kernel_blobs = KernelKMeans(n_clusters=3, kernel="linear", max_iter=100, random_state=42)
y_kernel_blobs = km_kernel_blobs.fit_predict(X_blobs)

# Plot Comparison
plot_clustering_results(X_blobs, y_blobs, y_std_blobs, y_kernel_blobs,
                        title="Experiment 1: Linearly Separable Blobs (Linear Kernel)",
                        kernel_name="Kernel K-Means (Linear)")

### 3.2 Experiment 2: Concentric Circles (Non-linear Cluster Structures)

<div dir="rtl">

دو دایره هم‌مرکز تو در تو تولید می‌کنیم. K-Means استاندارد مرزهای خوشه‌ای کروی فرض می‌کند و مجموعه‌داده را از وسط تقسیم می‌کند.

ما Kernel K-Means را با استفاده از **کرنل تابع پایه شعاعی (RBF / گاوسی)** اعمال می‌کنیم:

</div>

$$
K(x, y) = \exp\left( -\gamma \| x - y \|^2 \right)
$$

<div dir="rtl">

کرنل RBF مجموعه‌داده را به فضایی با بُعد بالاتر تصویر می‌کند که در آن حلقه‌ها در امتداد محور z (ارتفاع) به صورت خطی جداپذیر می‌شوند.

</div>

In [ ]:
# Generate Concentric Circles
X_circles, y_circles = make_circles(n_samples=400, factor=0.4, noise=0.05, random_state=42)

# Fit Standard K-Means
km_std_circles = KMeans(n_clusters=2, random_state=42, n_init='auto')
y_std_circles = km_std_circles.fit_predict(X_circles)

# Fit Kernel K-Means with RBF Kernel (gamma = 7.0)
km_kernel_circles = KernelKMeans(n_clusters=2, kernel="rbf", gamma=7.0, max_iter=100, random_state=42)
y_kernel_circles = km_kernel_circles.fit_predict(X_circles)

# Plot Comparison
plot_clustering_results(X_circles, y_circles, y_std_circles, y_kernel_circles,
                        title="Experiment 2: Concentric Circles (RBF Kernel)",
                        kernel_name="Kernel K-Means (RBF, gamma=7.0)")

### 3.3 Experiment 3: Interlocking Moons

<div dir="rtl">

دو نیم‌هلال درهم‌تنیده تولید می‌کنیم. K-Means استاندارد در اینجا نیز شکست می‌خورد. ما Kernel K-Means را با استفاده از **کرنل RBF** با ضریب مقیاس مناسب $\gamma = 12.0$ اجرا می‌کنیم.

</div>

In [ ]:
# Generate Moons
X_moons, y_moons = make_moons(n_samples=400, noise=0.05, random_state=42)

# Fit Standard K-Means
km_std_moons = KMeans(n_clusters=2, random_state=42, n_init='auto')
y_std_moons = km_std_moons.fit_predict(X_moons)

# Fit Kernel K-Means with RBF Kernel (gamma = 12.0)
km_kernel_moons = KernelKMeans(n_clusters=2, kernel="rbf", gamma=12.0, max_iter=100, random_state=42)
y_kernel_moons = km_kernel_moons.fit_predict(X_moons)

# Plot Comparison
plot_clustering_results(X_moons, y_moons, y_std_moons, y_kernel_moons,
                        title="Experiment 3: Interlocking Moons (RBF Kernel)",
                        kernel_name="Kernel K-Means (RBF, gamma=12.0)")

## 4. Hyperparameter Sensitivity: Effect of Gamma ($\gamma$) on RBF Clustering

<div dir="rtl">

عملکرد Kernel K-Means با کرنل RBF به شدت به مقیاس کرنل $\gamma$ حساس است:

</div>

<div dir="rtl">

- **$\gamma$ کوچک (مثلاً $0.1$)**: نگاشت بسیار شبیه به کرنل خطی عمل می‌کند. شباهت بین نقاط دور زیاد باقی می‌ماند، که باعث کم‌برازش و جداسازی خطی می‌شود.
- **$\gamma$ بهینه (مثلاً $10.0$)**: نقاط را به خوبی به ساختاری با جداسازی مناسب در فضای ویژگی تصویر می‌کند.
- **$\gamma$ بزرگ (مثلاً $150.0$)**: شباهت بین نقاط بسیار سریع کاهش می‌یابد. تقریباً تمام نقاط در فضای ویژگی متعامد می‌شوند ($K_{ij} \approx 0$ برای $i \neq j$، $K_{ii} = 1$). این منجر به بیش‌برازش، حساسیت به مقداردهی اولیه و خطاهای عددی مانند خوشه‌های خالی می‌شود.

</div>

In [ ]:
gammas = [0.1, 2.0, 15.0, 150.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, gamma in enumerate(gammas):
    try:
        km_gamma = KernelKMeans(n_clusters=2, kernel="rbf", gamma=gamma, random_state=42)
        y_gamma = km_gamma.fit_predict(X_moons)
        ari = adjusted_rand_score(y_moons, y_gamma)
        
        axes[idx].scatter(X_moons[:, 0], X_moons[:, 1], c=y_gamma, cmap='coolwarm', edgecolor='k', alpha=0.8, s=40)
        axes[idx].set_title(f"RBF Kernel with $\gamma = {gamma}$\nAdjusted Rand Index (ARI): {ari:.4f}", fontsize=12, fontweight='bold')
    except Exception as e:
        axes[idx].text(0.5, 0.5, f"Failed for $\gamma = {gamma}$:\n{str(e)}", 
                      ha='center', va='center', fontsize=12, color='red', weight='bold')
        axes[idx].set_title(f"RBF Kernel with $\gamma = {gamma}$ (Numerical Error)", fontsize=12, fontweight='bold')
        
    axes[idx].grid(True, linestyle='--', alpha=0.5)
    axes[idx].set_xlabel("Feature 1")
    axes[idx].set_ylabel("Feature 2")

plt.suptitle("Hyperparameter Sensitivity: Effect of Gamma on Moons Clustering", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Convergence Analysis

<div dir="rtl">

یکی از جنبه‌های مهم هر الگوریتم تکراری، بررسی همگرایی آن است. در اینجا تابع هدف Kernel K-Means (مجموع فاصله‌های درون‌خوشه‌ای) را در هر تکرار ثبت و رسم می‌کنیم تا نشان دهیم الگوریتم به درستی همگرا می‌شود.

</div>

In [ ]:
class KernelKMeansWithHistory(KernelKMeans):
    """
    Extended KernelKMeans that tracks the objective function value
    at each iteration for convergence analysis.
    """
    def fit(self, X, y=None, sample_weight=None):
        n_samples = X.shape[0]
        K = self._get_kernel(X)
        sw = sample_weight if sample_weight is not None else np.ones(n_samples)
        self.sample_weight_ = sw
        rs = check_random_state(self.random_state)
        self.labels_ = rs.randint(self.n_clusters, size=n_samples)
        dist = np.zeros((n_samples, self.n_clusters))
        self.within_distances_ = np.zeros(self.n_clusters)
        self.objective_history_ = []

        for it in range(self.max_iter):
            dist.fill(0)
            self._compute_dist(K, dist, self.within_distances_, update_within=True)
            # Compute objective: sum of min distances for each sample
            obj = sum(dist[i, self.labels_[i]] for i in range(n_samples))
            self.objective_history_.append(obj)
            labels_old = self.labels_
            self.labels_ = dist.argmin(axis=1)
            n_same = np.sum((self.labels_ - labels_old) == 0)
            if 1 - float(n_same) / n_samples < self.tol:
                break

        self.X_fit_ = X
        return self

# Run convergence analysis on Moons dataset
km_conv = KernelKMeansWithHistory(n_clusters=2, kernel='rbf', gamma=12.0,
                                   max_iter=50, tol=1e-4, random_state=42)
km_conv.fit(X_moons)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(km_conv.objective_history_) + 1), km_conv.objective_history_,
        'b-o', linewidth=2, markersize=6)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Objective Function (Within-Cluster Distance Sum)', fontsize=12)
ax.set_title('Convergence of Kernel K-Means (RBF, $\\gamma=12.0$) on Moons',
             fontsize=14, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Sensitivity to Initialization

<div dir="rtl">

Kernel K-Means مانند K-Means استاندارد به مقداردهی اولیه تصادفی حساس است. در اینجا الگوریتم را با ۱۰ مقداردهی اولیه مختلف اجرا می‌کنیم و توزیع ARI حاصل را بررسی می‌کنیم.

</div>

In [ ]:
# Test sensitivity to initialization with 10 different random seeds
n_runs = 10
ari_scores_circles = []
ari_scores_moons = []

for seed in range(n_runs):
    # Circles
    km_c = KernelKMeans(n_clusters=2, kernel='rbf', gamma=7.0,
                        max_iter=100, random_state=seed)
    y_pred_c = km_c.fit_predict(X_circles)
    ari_scores_circles.append(adjusted_rand_score(y_circles, y_pred_c))
    
    # Moons
    km_m = KernelKMeans(n_clusters=2, kernel='rbf', gamma=12.0,
                        max_iter=100, random_state=seed)
    y_pred_m = km_m.fit_predict(X_moons)
    ari_scores_moons.append(adjusted_rand_score(y_moons, y_pred_m))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(n_runs), ari_scores_circles, color='steelblue', edgecolor='k')
axes[0].axhline(y=np.mean(ari_scores_circles), color='red', linestyle='--',
               label=f'Mean ARI = {np.mean(ari_scores_circles):.4f}')
axes[0].set_xlabel('Random Seed', fontsize=11)
axes[0].set_ylabel('ARI', fontsize=11)
axes[0].set_title('Initialization Sensitivity - Circles\n'
                  f'(Std: {np.std(ari_scores_circles):.4f})',
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.4)
axes[0].set_ylim([0, 1.1])

axes[1].bar(range(n_runs), ari_scores_moons, color='coral', edgecolor='k')
axes[1].axhline(y=np.mean(ari_scores_moons), color='red', linestyle='--',
               label=f'Mean ARI = {np.mean(ari_scores_moons):.4f}')
axes[1].set_xlabel('Random Seed', fontsize=11)
axes[1].set_ylabel('ARI', fontsize=11)
axes[1].set_title('Initialization Sensitivity - Moons\n'
                  f'(Std: {np.std(ari_scores_moons):.4f})',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.4)
axes[1].set_ylim([0, 1.1])

plt.suptitle('Sensitivity to Random Initialization', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Circles - Mean ARI: {np.mean(ari_scores_circles):.4f}, '
      f'Std: {np.std(ari_scores_circles):.4f}')
print(f'Moons   - Mean ARI: {np.mean(ari_scores_moons):.4f}, '
      f'Std: {np.std(ari_scores_moons):.4f}')

## 7. Experiment with Polynomial Kernel

<div dir="rtl">

علاوه بر کرنل RBF، کرنل چندجمله‌ای نیز یکی از کرنل‌های پرکاربرد است:

</div>

$$
K(x, y) = (\gamma \cdot x^T y + c_0)^d
$$

<div dir="rtl">

در اینجا عملکرد کرنل چندجمله‌ای با درجات مختلف را روی دیتاست دایره‌های هم‌مرکز بررسی می‌کنیم.

</div>

In [ ]:
degrees = [2, 3, 4, 5]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, degree in enumerate(degrees):
    try:
        km_poly = KernelKMeans(n_clusters=2, kernel='poly', degree=degree,
                              gamma=1.0, coef0=1, max_iter=100, random_state=42)
        y_poly = km_poly.fit_predict(X_circles)
        ari = adjusted_rand_score(y_circles, y_poly)
        
        axes[idx].scatter(X_circles[:, 0], X_circles[:, 1], c=y_poly,
                         cmap='coolwarm', edgecolor='k', alpha=0.8, s=40)
        axes[idx].set_title(f'Polynomial Kernel (degree={degree})\n'
                           f'ARI: {ari:.4f}', fontsize=12, fontweight='bold')
    except Exception as e:
        axes[idx].text(0.5, 0.5, f'Failed for degree={degree}:\n{str(e)}',
                      ha='center', va='center', fontsize=11, color='red')
        axes[idx].set_title(f'Polynomial Kernel (degree={degree}) - Error',
                           fontsize=12, fontweight='bold')
    axes[idx].grid(True, linestyle='--', alpha=0.5)
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.suptitle('Polynomial Kernel with Different Degrees on Circles',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Comprehensive Evaluation Metrics

<div dir="rtl">

برای ارزیابی جامع‌تر، علاوه بر ARI و Silhouette Score، معیارهای NMI (اطلاعات متقابل نرمال‌شده) و Purity را نیز محاسبه می‌کنیم.

</div>

$$
\text{NMI}(Y, C) = \frac{2 \cdot I(Y; C)}{H(Y) + H(C)}
$$

$$
\text{Purity} = \frac{1}{N} \sum_{k=1}^{K} \max_j |C_k \cap T_j|
$$

<div dir="rtl">

که در آن $I(Y;C)$ اطلاعات متقابل، $H(\cdot)$ آنتروپی، $C_k$ خوشه $k$-ام و $T_j$ کلاس واقعی $j$-ام است.

</div>

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

def purity_score(y_true, y_pred):
    """Compute purity of clustering."""
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    return np.sum(np.amax(cm, axis=0)) / np.sum(cm)

# Evaluate all experiments with comprehensive metrics
datasets = {
    'Blobs (Linear)': (y_blobs, y_kernel_blobs),
    'Circles (RBF)': (y_circles, y_kernel_circles),
    'Moons (RBF)': (y_moons, y_kernel_moons)
}

print(f'{"Dataset":<20} {"ARI":<10} {"NMI":<10} {"Purity":<10} {"Silhouette":<10}')
print('=' * 60)

results = []
for name, (y_true, y_pred) in datasets.items():
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    purity = purity_score(y_true, y_pred)
    # Get corresponding X for silhouette
    if 'Blobs' in name:
        X_data = X_blobs
    elif 'Circles' in name:
        X_data = X_circles
    else:
        X_data = X_moons
    sil = silhouette_score(X_data, y_pred) if len(np.unique(y_pred)) > 1 else 0.0
    print(f'{name:<20} {ari:<10.4f} {nmi:<10.4f} {purity:<10.4f} {sil:<10.4f}')
    results.append([name, ari, nmi, purity, sil])

# Also compare with Standard K-Means
print('\n--- Standard K-Means for comparison ---')
print(f'{"Dataset":<20} {"ARI":<10} {"NMI":<10} {"Purity":<10} {"Silhouette":<10}')
print('=' * 60)

datasets_std = {
    'Blobs': (y_blobs, y_std_blobs, X_blobs),
    'Circles': (y_circles, y_std_circles, X_circles),
    'Moons': (y_moons, y_std_moons, X_moons)
}

for name, (y_true, y_pred, X_data) in datasets_std.items():
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    purity = purity_score(y_true, y_pred)
    sil = silhouette_score(X_data, y_pred) if len(np.unique(y_pred)) > 1 else 0.0
    print(f'{name:<20} {ari:<10.4f} {nmi:<10.4f} {purity:<10.4f} {sil:<10.4f}')

## 9. Comparison with Spectral Clustering

<div dir="rtl">

همان‌طور که در بخش بحث اشاره شد، ارتباط نظری عمیقی بین Kernel K-Means و خوشه‌بندی طیفی وجود دارد. در اینجا عملکرد هر دو روش را به صورت عملی مقایسه می‌کنیم.

</div>

In [ ]:
from sklearn.cluster import SpectralClustering

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Circles ---
# Ground Truth
axes[0, 0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles,
                   cmap='viridis', edgecolor='k', alpha=0.8, s=40)
axes[0, 0].set_title('Circles - Ground Truth', fontsize=12, fontweight='bold')

# Kernel K-Means
axes[0, 1].scatter(X_circles[:, 0], X_circles[:, 1], c=y_kernel_circles,
                   cmap='coolwarm', edgecolor='k', alpha=0.8, s=40)
ari_kkm_c = adjusted_rand_score(y_circles, y_kernel_circles)
axes[0, 1].set_title(f'Circles - Kernel K-Means\nARI: {ari_kkm_c:.4f}',
                     fontsize=12, fontweight='bold')

# Spectral Clustering
sc_circles = SpectralClustering(n_clusters=2, affinity='rbf', gamma=7.0,
                                random_state=42, assign_labels='kmeans')
y_sc_circles = sc_circles.fit_predict(X_circles)
ari_sc_c = adjusted_rand_score(y_circles, y_sc_circles)
axes[0, 2].scatter(X_circles[:, 0], X_circles[:, 1], c=y_sc_circles,
                   cmap='plasma', edgecolor='k', alpha=0.8, s=40)
axes[0, 2].set_title(f'Circles - Spectral Clustering\nARI: {ari_sc_c:.4f}',
                     fontsize=12, fontweight='bold')

# --- Moons ---
# Ground Truth
axes[1, 0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons,
                   cmap='viridis', edgecolor='k', alpha=0.8, s=40)
axes[1, 0].set_title('Moons - Ground Truth', fontsize=12, fontweight='bold')

# Kernel K-Means
axes[1, 1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_kernel_moons,
                   cmap='coolwarm', edgecolor='k', alpha=0.8, s=40)
ari_kkm_m = adjusted_rand_score(y_moons, y_kernel_moons)
axes[1, 1].set_title(f'Moons - Kernel K-Means\nARI: {ari_kkm_m:.4f}',
                     fontsize=12, fontweight='bold')

# Spectral Clustering
sc_moons = SpectralClustering(n_clusters=2, affinity='rbf', gamma=12.0,
                              random_state=42, assign_labels='kmeans')
y_sc_moons = sc_moons.fit_predict(X_moons)
ari_sc_m = adjusted_rand_score(y_moons, y_sc_moons)
axes[1, 2].scatter(X_moons[:, 0], X_moons[:, 1], c=y_sc_moons,
                   cmap='plasma', edgecolor='k', alpha=0.8, s=40)
axes[1, 2].set_title(f'Moons - Spectral Clustering\nARI: {ari_sc_m:.4f}',
                     fontsize=12, fontweight='bold')

for ax in axes.ravel():
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Kernel K-Means vs Spectral Clustering',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Runtime Scalability Analysis

<div dir="rtl">

در اینجا زمان اجرای K-Means استاندارد و Kernel K-Means را با افزایش تعداد نمونه‌ها ($N$) اندازه‌گیری می‌کنیم تا تفاوت پیچیدگی محاسباتی $\mathcal{O}(N \cdot K)$ در مقابل $\mathcal{O}(N^2 \cdot K)$ را به صورت عملی نشان دهیم.

</div>

In [ ]:
import time

sample_sizes = [100, 200, 500, 1000, 2000, 3000]
times_std = []
times_kernel = []

for n in sample_sizes:
    X_test, _ = make_moons(n_samples=n, noise=0.05, random_state=42)
    
    # Standard K-Means
    start = time.time()
    KMeans(n_clusters=2, random_state=42, n_init='auto').fit(X_test)
    times_std.append(time.time() - start)
    
    # Kernel K-Means
    start = time.time()
    KernelKMeans(n_clusters=2, kernel='rbf', gamma=12.0,
                 max_iter=100, random_state=42).fit(X_test)
    times_kernel.append(time.time() - start)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(sample_sizes, times_std, 'b-o', linewidth=2, markersize=7, label='Standard K-Means')
ax.plot(sample_sizes, times_kernel, 'r-s', linewidth=2, markersize=7, label='Kernel K-Means (RBF)')
ax.set_xlabel('Number of Samples (N)', fontsize=12)
ax.set_ylabel('Runtime (seconds)', fontsize=12)
ax.set_title('Runtime Scalability: Standard K-Means vs Kernel K-Means',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('\nRuntime Summary:')
print(f'{"N":<8} {"Std K-Means (s)":<18} {"Kernel K-Means (s)":<20} {"Ratio":<8}')
print('=' * 55)
for i, n in enumerate(sample_sizes):
    ratio = times_kernel[i] / times_std[i] if times_std[i] > 0 else float('inf')
    print(f'{n:<8} {times_std[i]:<18.4f} {times_kernel[i]:<20.4f} {ratio:<8.1f}x')

## 11. Interpretation of Experimental Results

<div dir="rtl">

### تفسیر آزمایش ۱ (Blobs):
هر دو الگوریتم K-Means استاندارد و Kernel K-Means با کرنل خطی عملکرد تقریباً یکسانی دارند. این نتیجه مورد انتظار است زیرا کرنل خطی $K(x,y) = x^Ty$ معادل نگاشت همانی $\Phi(x) = x$ است و داده‌ها از قبل به صورت خطی جداپذیر هستند.

### تفسیر آزمایش ۲ (Circles):
K-Means استاندارد داده‌ها را با یک خط عمودی تقسیم می‌کند (ARI نزدیک صفر) زیرا فقط مرزهای خطی/کروی ایجاد می‌کند. اما Kernel K-Means با کرنل RBF و $\gamma=7.0$ به درستی دو حلقه هم‌مرکز را شناسایی می‌کند (ARI نزدیک ۱). دلیل این موفقیت آن است که کرنل RBF داده‌ها را به فضایی با بُعد بالاتر نگاشت می‌کند که در آن حلقه‌ها به صورت خطی جداپذیر می‌شوند.

### تفسیر آزمایش ۳ (Moons):
مشابه آزمایش ۲، K-Means استاندارد نمی‌تواند ساختار غیرخطی هلال‌ها را تشخیص دهد. Kernel K-Means با $\gamma=12.0$ (مقدار بزرگ‌تر نسبت به دایره‌ها، زیرا هلال‌ها ساختار پیچیده‌تری دارند) به خوبی دو هلال را جدا می‌کند.

### تفسیر تحلیل حساسیت $\gamma$:
- $\gamma$ خیلی کوچک: کرنل RBF تقریباً مانند کرنل خطی عمل می‌کند → کم‌برازش
- $\gamma$ بهینه: جداسازی مناسب در فضای ویژگی → بهترین عملکرد
- $\gamma$ خیلی بزرگ: هر نقطه تقریباً فقط با خودش شباهت دارد → بیش‌برازش و ناپایداری عددی

این رفتار مشابه trade-off بین bias و variance در یادگیری نظارت‌شده است.

</div>

## 12. Kernel Matrix Visualization

<div dir="rtl">

برای درک بهتر نحوه عملکرد کرنل‌های مختلف، ماتریس کرنل $K$ را برای دیتاست دایره‌ها با مقادیر مختلف $\gamma$ تجسم می‌کنیم. در ماتریس کرنل، هر عنصر $K_{ij}$ نشان‌دهنده شباهت بین نمونه‌های $x_i$ و $x_j$ در فضای ویژگی است.

</div>

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

# Sort data by true labels for better visualization
sort_idx = np.argsort(y_circles)
X_sorted = X_circles[sort_idx]

gammas_viz = [0.1, 2.0, 7.0, 50.0]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for idx, gamma in enumerate(gammas_viz):
    K_matrix = rbf_kernel(X_sorted, gamma=gamma)
    im = axes[idx].imshow(K_matrix, cmap='hot', interpolation='nearest')
    axes[idx].set_title(f'$\\gamma = {gamma}$', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Sample Index')
    axes[idx].set_ylabel('Sample Index')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

plt.suptitle('RBF Kernel Matrix Visualization (Circles, sorted by label)',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 13. Discussion & SML Perspective

### 13.1 Complexity Analysis
- **Standard K-Means**:
  - **Time Complexity**: $\mathcal{O}(I \cdot N \cdot d \cdot K)$
  - **Space Complexity**: $\mathcal{O}(N \cdot d)$
  - <div dir="rtl">**مقیاس‌پذیری**: بسیار مقیاس‌پذیر؛ معمولاً روی مجموعه‌داده‌های بزرگ استفاده می‌شود.</div>
- **Kernel K-Means**:
  - **Time Complexity**: $\mathcal{O}(I \cdot N^2 \cdot K)$
  - **Space Complexity**: $\mathcal{O}(N^2)$
  - <div dir="rtl">**مقیاس‌پذیری**: برای مجموعه‌داده‌های بسیار بزرگ ($N > 10,000$) مقیاس‌پذیر نیست. برای مجموعه‌داده‌های بزرگ‌تر، باید از تقریب‌های کرنل مانند **تقریب Nyström** یا **ویژگی‌های فوریه تصادفی (RFF)** برای نگاشت صریح داده‌ها به فضایی با بُعد پایین استفاده کرد.</div>

<div dir="rtl">

که در آن $I$ تعداد تکرارها، $N$ تعداد نمونه‌ها، $d$ تعداد ویژگی‌ها و $K$ تعداد خوشه‌ها است.

</div>

### 13.2 Relationship with Spectral Clustering
<div dir="rtl">

ارتباط عمیقی بین **Kernel K-Means** و **خوشه‌بندی طیفی** وجود دارد:

- خوشه‌بندی طیفی داده‌ها را با استفاده از بردارهای ویژه ماتریس لاپلاسین گراف نگاشت می‌کند و سپس K-means استاندارد را در فضای تعبیه طیفی اعمال می‌کند.
- از نظر ریاضی، می‌توان اثبات کرد که تابع هدف **Weighted Kernel K-Means** دقیقاً معادل اهداف ratio association و normalized cut (NCut) در افراز طیفی گراف است، زمانی که کرنل بر حسب ماتریس وابستگی گراف تعریف شود.
- همان‌طور که در بخش ۹ مشاهده شد، هر دو روش عملکرد مشابهی دارند، اما خوشه‌بندی طیفی معمولاً پایدارتر است زیرا از تجزیه ویژه‌مقداری استفاده می‌کند که وابسته به مقداردهی اولیه تصادفی نیست.

</div>

### 13.3 Summary of Strengths and Limitations

| Feature / Metric | Standard K-Means | Kernel K-Means |
| :--- | :--- | :--- |
| **Cluster Boundaries** | Linear / Spherical Voronoi Cells | Complex, Non-linear, Topological |
| **Computational Cost** | $\mathcal{O}(I \cdot N \cdot d \cdot K)$ | $\mathcal{O}(I \cdot N^2 \cdot K)$ |
| **Memory Requirements** | $\mathcal{O}(N \cdot d)$ | $\mathcal{O}(N^2)$ (stores kernel matrix) |
| **Hyperparameters** | Only $K$ (number of clusters) | $K$, Kernel Type, and Kernel parameters (e.g., $\gamma, d, c_0$) |
| **Interpretability** | Easy (centroids are physical data coordinates) | Difficult (centroids are in implicit infinite-dimensional space $\mathcal{H}$) |
| **Initialization Sensitivity** | Moderate (K-Means++ helps) | High (no equivalent of K-Means++) |
| **Scalability** | Excellent ($N > 10^6$) | Poor ($N < 10^4$) |
